<a href="https://colab.research.google.com/github/hohohoo07/Big_Data_Processing_and_Applications/blob/main/KW_MMDS_Colab_1_(WordCount).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# KW_MMDS - Colab 1 (Hints)
## Wordcount in Spark

### Setup

Let's setup Spark on your Colab environment.  Run the cell below!

In [ ]:
!pip install -U -q PyDrive2
import os

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.0/48.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 2.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
joserfc 1.7.5 requires cryptography>=45.0.1, but you have cryptography 43.0.3 which is incompatible.
authlib 1.8.0 requires cryptography>=45.0.1, but you have cryptography 43.0.3 which is incompatible.


Now we authenticate a Google Drive client to download the file we will be processing in our Spark job.

**Make sure to follow the interactive instructions.**

In [ ]:
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

# Authenticate and create the PyDrive client
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

In [ ]:
# 사용할 파일 'pg100.txt' 다운로드
id='1SE6k_0YukzGd5wK-E4i6mG83nydlfvSa'
downloaded = drive.CreateFile({'id': id})
downloaded.GetContentFile('pg100.txt')

If you executed the cells above, you should be able to see the file *pg100.txt* under the "Files" tab on the left panel.

### Your task

If you run successfully the setup stage, you are ready to work on the *pg100.txt* file which contains a copy of the complete works of Shakespeare.

Write a Spark application which outputs the number of words that start with each letter. This means that for every letter we want to count the total number of (non-unique) words that start with a specific letter. In your implementation **ignore the letter case**, i.e., consider all words as lower case. Also, you can ignore all the words **starting** with a non-alphabetic character.

(셰익스피어 전집에서 a부터 z까지 각 알파벳으로 시작하는 단어의 수를 세어보세요. 대문자는 소문자로 해석합니다. 또한 알파벳으로 시작하지 않는 단어는 무시합니다.)

In [ ]:
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark import SparkContext
import pandas as pd

# create the Spark Session
spark = SparkSession.builder.getOrCreate()

# create the Spark Context
sc = spark.sparkContext

In [ ]:
# pg100.txt 파일 RDD로 읽기
# sc.textFile은 HDFS로부터 text file 한 줄, 한 줄을 RDD로 반환
RDDs = sc.textFile('pg100.txt')
RDDs.take(10)

['The Project Gutenberg EBook of The Complete Works of William Shakespeare, by',
 'William Shakespeare',
 '',
 'This eBook is for the use of anyone anywhere at no cost and with',
 'almost no restrictions whatsoever.  You may copy it, give it away or',
 're-use it under the terms of the Project Gutenberg License included',
 'with this eBook or online at www.gutenberg.org',
 '',
 '** This is a COPYRIGHTED Project Gutenberg eBook, Details Below **',
 '**     Please follow the copyright guidelines in this file.     **']

In [ ]:
# 소문자로 바꾸기
# 무명함수 lambda 이용, line은 이름 바꿔서 사용해도 됨
RDDs = RDDs.map(lambda line: line.lower())
RDDs.take(10)

['the project gutenberg ebook of the complete works of william shakespeare, by',
 'william shakespeare',
 '',
 'this ebook is for the use of anyone anywhere at no cost and with',
 'almost no restrictions whatsoever.  you may copy it, give it away or',
 're-use it under the terms of the project gutenberg license included',
 'with this ebook or online at www.gutenberg.org',
 '',
 '** this is a copyrighted project gutenberg ebook, details below **',
 '**     please follow the copyright guidelines in this file.     **']

In [ ]:
# 단어별로 나누기
# map으로 하면 split된 결과가 list로 나와서 단어 각각을 RDD로 가지는 게 아니라
# 아직도 한 문장이 하나의 RDD라는 문제 존재
# 따라서 flatMap 이용 -> 단어의 개수만큼 RDD 생성됨
RDDs = RDDs.flatMap(lambda line: line.split(" "))
RDDs.take(20)

['the',
 'project',
 'gutenberg',
 'ebook',
 'of',
 'the',
 'complete',
 'works',
 'of',
 'william',
 'shakespeare,',
 'by',
 'william',
 'shakespeare',
 '',
 'this',
 'ebook',
 'is',
 'for',
 'the']

In [ ]:
# 공백 제거하기
RDDs = RDDs.filter(lambda word: len(word) > 0)
RDDs.take(60)

['the',
 'project',
 'gutenberg',
 'ebook',
 'of',
 'the',
 'complete',
 'works',
 'of',
 'william',
 'shakespeare,',
 'by',
 'william',
 'shakespeare',
 'this',
 'ebook',
 'is',
 'for',
 'the',
 'use',
 'of',
 'anyone',
 'anywhere',
 'at',
 'no',
 'cost',
 'and',
 'with',
 'almost',
 'no',
 'restrictions',
 'whatsoever.',
 'you',
 'may',
 'copy',
 'it,',
 'give',
 'it',
 'away',
 'or',
 're-use',
 'it',
 'under',
 'the',
 'terms',
 'of',
 'the',
 'project',
 'gutenberg',
 'license',
 'included',
 'with',
 'this',
 'ebook',
 'or',
 'online',
 'at',
 'www.gutenberg.org',
 '**',
 'this']

In [ ]:
# 알파벳으로 시작하는 단어만 남기기 filter()
RDDs = RDDs.filter(lambda word: word[0].isalpha())
RDDs.take(60)

['the',
 'project',
 'gutenberg',
 'ebook',
 'of',
 'the',
 'complete',
 'works',
 'of',
 'william',
 'shakespeare,',
 'by',
 'william',
 'shakespeare',
 'this',
 'ebook',
 'is',
 'for',
 'the',
 'use',
 'of',
 'anyone',
 'anywhere',
 'at',
 'no',
 'cost',
 'and',
 'with',
 'almost',
 'no',
 'restrictions',
 'whatsoever.',
 'you',
 'may',
 'copy',
 'it,',
 'give',
 'it',
 'away',
 'or',
 're-use',
 'it',
 'under',
 'the',
 'terms',
 'of',
 'the',
 'project',
 'gutenberg',
 'license',
 'included',
 'with',
 'this',
 'ebook',
 'or',
 'online',
 'at',
 'www.gutenberg.org',
 'this',
 'is']

In [ ]:
# (word, 1) tuples 만들기 map()
RDDs = RDDs.map(lambda word: (word, 1))
RDDs.take(10)

[('the', 1),
 ('project', 1),
 ('gutenberg', 1),
 ('ebook', 1),
 ('of', 1),
 ('the', 1),
 ('complete', 1),
 ('works', 1),
 ('of', 1),
 ('william', 1)]

In [ ]:
# Reduce
# key 값이 같은 두 RDD의 value를 더해야 하니까
# 람다함수 a+b !!
RDDs = RDDs.reduceByKey(lambda a, b: a+b)
RDDs.take(10)

[('of', 18126),
 ('works', 268),
 ('william', 311),
 ('shakespeare,', 2),
 ('by', 4310),
 ('shakespeare', 270),
 ('this', 5930),
 ('for', 8000),
 ('use', 509),
 ('anyone', 5)]

In [ ]:
# 정렬하기 위해 key, value 위치 바꾸기
RDDs = RDDs.map(lambda tuple: (tuple[1], tuple[0]))
RDDs.take(10)

[(18126, 'of'),
 (268, 'works'),
 (311, 'william'),
 (2, 'shakespeare,'),
 (4310, 'by'),
 (270, 'shakespeare'),
 (5930, 'this'),
 (8000, 'for'),
 (509, 'use'),
 (5, 'anyone')]

In [ ]:
# 내림차순으로 정렬하기 sortByKey
# sortByKey는 기본으로 Ascending
# 내림차순으로 정렬하려면 ascending=False!!
RDDs = RDDs.sortByKey(ascending=False)
RDDs.take(10)

[(27730, 'the'),
 (26099, 'and'),
 (19540, 'i'),
 (18762, 'to'),
 (18126, 'of'),
 (14436, 'a'),
 (12456, 'my'),
 (10730, 'in'),
 (10696, 'you'),
 (10501, 'that')]

In [ ]:
# 한 번에 몰아서 하기
# RDDs = sc.textFile('pg100.txt').map(lambda line: line.lower())\
#                               .flatMap(lambda line: line.split(" "))\
#                               .filter(lambda word: len(word) > 0)\
#                               .?(lambda word: word[0].isalpha())\
#                               .?(lambda word: (word, 1))\
#                               .reduceByKey(lambda a, b: )\
#                               .map(lambda tuple: (tuple[1], tuple[0]))\
#                               .sortByKey(=)
RDDs = sc.textFile('pg100.txt').map(lambda line: line.lower())\
                              .flatMap(lambda line: line.split(" "))\
                              .filter(lambda word: len(word) > 0)\
                              .filter(lambda word: word[0].isalpha())\
                              .map(lambda word: (word, 1))\
                              .reduceByKey(lambda a, b: a+b)\
                              .map(lambda tuple: (tuple[1], tuple[0]))\
                              .sortByKey(ascending=False)
RDDs.take(10)

[(27730, 'the'),
 (26099, 'and'),
 (19540, 'i'),
 (18762, 'to'),
 (18126, 'of'),
 (14436, 'a'),
 (12456, 'my'),
 (10730, 'in'),
 (10696, 'you'),
 (10501, 'that')]